In [7]:
from model.SAC import Encoder, Q_network, V_network, P_network, ActionDecoder
import numpy as np
import torch

import json 


In [2]:
json_path = '/mnt/d/ML/Kaggle/StarWars/episode-76554885.json'
with open(json_path, 'r') as f:
    logs = json.load(f)
    print(logs.keys())


dict_keys(['configuration', 'description', 'id', 'info', 'module_version', 'name', 'rewards', 'schema_version', 'specification', 'statuses', 'steps', 'title', 'version'])


In [3]:
idx = 66
fleets = np.array(logs['steps'][idx][0]['observation']['fleets'])
planets = np.array(logs['steps'][idx][0]['observation']['planets'])
comet_planet_ids = np.array(logs['steps'][idx][0]['observation']['comet_planet_ids'])
angular_velocity = logs['steps'][idx][0]['observation']['angular_velocity']
initial_planets = np.array(logs['steps'][idx][0]['observation']['initial_planets'])

In [4]:
encoder = Encoder()
q_network = Q_network()
state = encoder(planets, fleets, initial_planets, angular_velocity, comet_planet_ids, time_step=120)
action = torch.zeros(5, q_network.d_model)  # dummy actions, shape: [num_actions, d_model]
value = q_network(state, action)
value.shape

/home/tan/miniconda3/envs/arc/lib/python3.10/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


torch.Size([1, 1])

In [8]:
v_network = V_network()
value = v_network(state)
value

/home/tan/miniconda3/envs/arc/lib/python3.10/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


tensor([[-0.6057]], grad_fn=<AddmmBackward0>)

In [9]:
policy = P_network()
mu, sigma = policy(state)
mu.shape, sigma.shape

(torch.Size([92, 8]), torch.Size([92, 8]))

In [11]:
action_decoder = ActionDecoder(d_action=8)
action = action_decoder(mu, sigma)
action.shape

torch.Size([92, 92])